# How to fine-tune HobbyLM-Embed on your own data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fuellabs-ai/cookbooks/blob/main/fine-tune-on-your-data/walkthrough.ipynb)

This notebook fine-tunes `harims95/hobbylm-embed-140m` on your own
(question, answer passage) pairs with MultipleNegativesRankingLoss, on a
laptop CPU, and saves a model folder you can load and use like the base one.
Always compare the fine-tuned model against the base model on your own
held-out data before switching; `evaluate.py` in this folder does that.


## Setup

The cell below clones the cookbook repo and moves into this folder. If the
notebook already sits next to `finetune.py`, for example in a local clone,
nothing is downloaded.


In [1]:
import os
import subprocess
from pathlib import Path

REPO = "https://github.com/fuellabs-ai/cookbooks.git"
FOLDER = "fine-tune-on-your-data"

if not Path("finetune.py").exists():
    if not Path("cookbooks/.git").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO, "cookbooks"], check=True)
    os.chdir(Path("cookbooks") / FOLDER)
print("cwd:", Path.cwd())

## Dependencies

The recipe was measured with sentence-transformers 6.3.0, so that version is
pinned. Training also needs accelerate, and the pairs CSV is read into a
small `datasets.Dataset`.


In [2]:
import importlib
import importlib.metadata
import subprocess
import sys

def ensure(spec, module):
    if spec.startswith("sentence-transformers"):
        if importlib.util.find_spec(module) is None or importlib.metadata.version("sentence-transformers") != "6.3.0":
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", spec], check=True)
    elif importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", spec], check=True)

for spec, module in [
    ("sentence-transformers==6.3.0", "sentence_transformers"),
    ("accelerate>=1.1.0", "accelerate"),
    ("datasets", "datasets"),
]:
    ensure(spec, module)
print("ready")

ready


## The sample data

Fictional sample data for a made-up company's help wiki: 100 train pairs, 20
held-out pairs for the comparison, and a corpus of 170 passages (120 gold +
50 distractors). Replace the three CSVs with your own data and keep the
column names: the trainer applies the model's query and passage prompts by
column name, so the prefixes stay part of the model.


In [3]:
import csv

for path in ("examples/train_pairs.csv", "examples/test_pairs.csv", "examples/passages.csv"):
    rows = list(csv.DictReader(open(path)))
    print(f"{path}: {len(rows)} rows")
    first = rows[0]
    print("  ", {k: (v[:58] + "...") if len(v) > 60 else v for k, v in first.items()})

examples/train_pairs.csv: 100 rows
   {'query': 'vpn keeps dropping every few minutes', 'passage': 'The corporate VPN client reconnects automatically when the...'}
examples/test_pairs.csv: 20 rows
   {'query': 'checking slack from the beach house, is that ok', 'passage_id': 'P-101'}
examples/passages.csv: 170 rows
   {'passage_id': 'P-001', 'passage': 'The corporate VPN client reconnects automatically when the...'}


## Load the base model

Both scripts pin the base model to revision
`ea708cee107818fa0c1871eebd26d82a1e3fb1aa` and load it with
`trust_remote_code=True`. This cell loads it the same way, which also warms
the cache for training. Questions are embedded with `prompt_name="query"`,
passages with `prompt_name="passage"`; the model adds its prefixes itself.


In [4]:
import warnings

warnings.filterwarnings("ignore")
import transformers

transformers.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()

from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "harims95/hobbylm-embed-140m",
    revision="ea708cee107818fa0c1871eebd26d82a1e3fb1aa",
    trust_remote_code=True,
)
print(type(model).__name__, "| prompts:", model.prompts)

SentenceTransformer | prompts: {'query': 'query: ', 'document': 'passage: ', 'passage': 'passage: '}


## Train

3 epochs, batch size 8, learning rate 2e-5, CPU. On the sample data that is
39 steps and about a minute and a half. The result is saved to
`my-hobbylm-embed/`.


In [ ]:
!{sys.executable} finetune.py

## Ask the saved model a question

Retrieve from the sample corpus with your own wording. Vectors are unit-norm,
so a dot product is cosine similarity.


In [6]:
model = SentenceTransformer("my-hobbylm-embed", trust_remote_code=True)

question = "how do i get a parking spot"
rows = list(csv.DictReader(open("examples/passages.csv")))
q = model.encode([question], prompt_name="query", convert_to_numpy=True)
p = model.encode([r["passage"] for r in rows], prompt_name="passage", convert_to_numpy=True)
sims = (q @ p.T)[0]

print("question:", question)
for idx in sims.argsort()[::-1][:3]:
    print(f"  {sims[idx]:.3f}  {rows[idx]['passage_id']}  {rows[idx]['passage'][:72]}...")

question: how do i get a parking spot
  0.713  P-050  Parking is allocated by the transport portal in order of requests, with ...
  0.664  D-025  The visitor parking spots P1 to P4 are booked in the transport portal by...
  0.501  D-016  The company kindergarten places are allocated each spring through the be...


## Compare against the base model before switching

Run `evaluate.py` to score both models on the held-out pairs:

```bash
python evaluate.py
```

Always compare the fine-tuned model against the base model on your own
held-out data before switching. If the comparison does not go your way, keep
the base model.

## Keep it

`my-hobbylm-embed/` is a complete Sentence Transformers model. Load it like
the base one and keep the same prompt names:

```python
model = SentenceTransformer("my-hobbylm-embed", trust_remote_code=True)
queries = model.encode([...], prompt_name="query")
passages = model.encode([...], prompt_name="passage")
```

Re-embed your whole corpus with the model you ship, and keep vectors from
base and fine-tuned models in separate indexes.


In [7]:
import shutil

shutil.make_archive("my-hobbylm-embed", "zip", "my-hobbylm-embed")
print("wrote my-hobbylm-embed.zip")

wrote my-hobbylm-embed.zip


## Where this breaks

- Small data fits itself: the model can fit the pairs you train on without
  carrying over to new questions. Keep a held-out split you never train on.
- A handful of held-out pairs is a small measuring tape; use 50 or more
  before trusting a small difference either way.
- Never mix vectors from base and fine-tuned models in one index; re-embed
  the whole corpus with the model you ship.
- Keep `prompt_name="query"` / `prompt_name="passage"` at serving time, or
  you serve a different task than you trained.

The README in this folder has the full how-to.
